[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DJ-lezama/grad-cam-music-similarity/blob/main/notebooks/walkthrough.ipynb)

# From Versionness to Infringement?
### A Grad-CAM study of audio-based musical similarity models

This notebook walks through the project end to end using the `musicsim` package in `src/`.
It started as a Fall 2025 course project (see the [poster](../docs/poster/poster.pdf)) and was extended in 2026;
the headline numbers below are the **current** results with the released checkpoint, and the course-version results
are shown alongside for comparison.

| Section | What it does | Needs |
|---|---|---|
| 1. Visual baseline | Classical CV on spectrograms (SSIM, edges, Hough, DTW) | two audio files |
| 2. Data | MelodySim → 3-channel spectrograms | `scripts/download_melodysim.py`, `scripts/preprocess_melodysim.py` |
| 3. Model | Siamese CNN | – |
| 4. MelodySim evaluation | ROC, confusion matrix, channel ablation on the held-out test split | data + checkpoint |
| 5. MCIC benchmark | real copyright cases | MCIC data + checkpoint |
| 6. Grad-CAM | where the model looks | data + checkpoint |

Saved figures are embedded, so the notebook reads well without running anything.
Cells that need data check for it first and skip with a message if it is missing.

In [ ]:
# Colab setup (skipped when running locally from a clone of the repo)
import os, sys
if "google.colab" in sys.modules and not os.path.exists("src/musicsim"):
    !git clone -q https://github.com/DJ-lezama/grad-cam-music-similarity.git
    %cd grad-cam-music-similarity
    !pip install -q -e .
    !mkdir -p checkpoints && wget -q -O checkpoints/siamese_best.pt https://github.com/DJ-lezama/grad-cam-music-similarity/releases/latest/download/siamese_best.pt
    # Optional: keep downloaded data on Google Drive between sessions
    # from google.colab import drive; drive.mount("/content/drive")
    # os.environ["MUSICSIM_DATA"] = "/content/drive/MyDrive/musicsim_data"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt, torch

from musicsim import config as C, plots
from musicsim.model import SiameseNet, load_model, get_device

device = get_device()
HAVE_CKPT = C.CHECKPOINT.exists()
HAVE_DATA = C.MANIFEST.exists()
HAVE_MCIC = C.MCIC_CSV.exists()
print(f"device={device}  checkpoint={HAVE_CKPT}  melodysim={HAVE_DATA}  mcic={HAVE_MCIC}")

## 1. Visual baseline

Before training anything: can two versions of a song be matched by treating their spectrograms as images?
The harmonic view (median filtering along time after CLAHE) keeps horizontal pitch ridges; Hough lines summarise them;
SSIM and multi-dimensional DTW give a similarity score.

![visual baseline](../docs/figures/fig1_visual_baseline.png)

The example pair is MCIC case 2 (*Ma Este Meg Boldog Vagyok* vs *Perhaps*). These hand-made views show structure, but give
no learned notion of which differences matter, which motivates the learned model below.

In [ ]:
from musicsim import visual_baseline as vb

a, b = C.MCIC_AUDIO_DIR / "002_a_Ma_Este_Meg_Boldog_Vagyok.wav", C.MCIC_AUDIO_DIR / "002_b_Perhaps.wav"
if a.exists() and b.exists():
    res = vb.compare(str(a), str(b))
    vb.plot(res, names=("A", "B")); plt.show()
else:
    print("MCIC audio not found; showing the saved figure above.")

## 2. Data: MelodySim → 3-channel spectrograms

[MelodySim](https://huggingface.co/datasets/amaai-lab/melodySim) (Lu et al., 2025) contains short segments (~9 s) of tracks in an
`original` rendition plus three re-arranged versions (`version_0..2`). The original run used 103 of its 539 training shards
(listed in `data/melodysim_shards.txt`; ~50 GB). The train / validation / test split of a segment comes from its
`relative_path`, so those shards contain all three splits.

Each segment becomes a `(3, 128, T)` array:

| channel | content |
|---|---|
| 0 | log-mel spectrogram (128 mels, 16 kHz, n_fft 1024, hop 256), min-max normalised |
| 1 | harmonic part of HPSS |
| 2 | percussive part of HPSS |

Clips are center-cropped / zero-padded to `T = 685` frames (~11 s, the 95th percentile of training lengths).

```bash
python scripts/download_melodysim.py --quick 3   # or no flag for all 103 shards
python scripts/preprocess_melodysim.py
```

In [ ]:
from musicsim.melodysim import load_manifest, split_frames
import pandas as pd

ref = pd.read_csv(C.REFERENCE_MANIFEST, dtype={"chunk_id": str})
print("Segments in the original run's manifest:")
print(ref.groupby("split").agg(segments=("track_id", "size"), tracks=("track_id", "nunique")))

if HAVE_DATA:
    splits = split_frames(load_manifest())
    ex = np.load(splits["test"]["path"].iloc[0]).astype(np.float32)
    fig, axes = plt.subplots(1, 3, figsize=(15, 3), constrained_layout=True)
    for ax, ch, name in zip(axes, ex, C.CHANNEL_NAMES):
        ax.imshow(ch, aspect="auto", origin="lower", cmap="magma"); ax.set_title(name)
    plt.show()

### How pairs are built

* **Positive (label 1):** the *same segment* (same track, same chunk) in two *different versions*.
* **Negative (label 0):** segments from two different tracks.

This definition is the crux of the study: the model is rewarded for recognising **version identity**, i.e. that two
renditions are the same piece, which is only the most extreme case of musical similarity.

## 3. Model

A shared 4-block CNN encoder (16→32→64→128 channels, global average pooling, 256-d L2-normalised embedding).
The two embeddings are compared with `|z_a − z_b|` and a small MLP outputs a similarity logit. Trained with binary
cross-entropy, Adam (lr 1e-4), ReduceLROnPlateau and early stopping on validation loss (`scripts/train.py`).

In [ ]:
model = SiameseNet()
print(model)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 4. MelodySim evaluation

**Current result (held-out test split, 9,751 fixed pairs, 100 unseen tracks):** ROC-AUC **0.758**, accuracy **0.684**.

**Channel ablation** (channels zeroed at inference time, no retraining). All three channels together clearly beat any
single channel or pair:

<img src="../docs/figures/test_melodysim_roc.png" width="300"> <img src="../docs/figures/test_ablation.png" width="470">

<details><summary>Course version (validation split, re-sampled pairs): ROC-AUC 0.833, accuracy 0.732</summary>

The course notebook reported these on the validation split, which was also used for early stopping, so they are
optimistic.

<img src="../docs/figures/fig2_melodysim_roc.png" width="300"> <img src="../docs/figures/fig2_ablation.png" width="430">
</details>

The cell below recomputes the current numbers with fixed, seeded pairs.

In [ ]:
from musicsim.evaluate import predict, metrics, ablation
from musicsim.pairs import FixedPairs

if HAVE_CKPT and HAVE_DATA:
    model = load_model(C.CHECKPOINT, device)
    test_pairs = FixedPairs(splits["test"])
    y, p = predict(model, test_pairs)
    m = metrics(y, p)
    print({k: round(v, 3) for k, v in m.items() if isinstance(v, float)})
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
    plots.roc(y, p, "MelodySim test", axes[0]); plots.confusion(m["confusion"], ax=axes[1]); plt.show()
    plots.ablation_bars(ablation(model, test_pairs)); plt.show()
else:
    print("Needs the checkpoint (see README) and preprocessed MelodySim data.")

## 5. MCIC benchmark: real copyright cases

MCIC (Park et al., ISMIR 2024) lists music-copyright court cases with the two disputed songs as MIDI and the court's
decision. 96 cases with a clear 0/1 decision can be rendered and scored.

**Current result:** ROC-AUC **0.450** using the middle ~11 s of each song, and **0.397** using the best-matching ~11 s
windows (`max-window`). Every pair scores between 0.61 and 0.87, whatever the court decided: the model treats any two
MIDI renderings (same soundfont) as versions of each other.

<img src="../docs/figures/test_mcic_score_distribution.png" width="780">

<details><summary>Course version: 95 cases, ROC-AUC 0.442</summary>

<img src="../docs/figures/fig3_mcic_roc.png" width="300"> <img src="../docs/figures/fig3_mcic_confusion.png" width="300">
</details>

In [ ]:
from musicsim.mcic import MCICAudio, load_pairs, score_pair, song_multispec

if HAVE_CKPT and HAVE_MCIC:
    pairs = load_pairs()
    audio = MCICAudio()
    rows = []
    for r in pairs.itertuples():
        try:
            a, b = song_multispec(audio.load(r.File_A)), song_multispec(audio.load(r.File_B))
        except (FileNotFoundError, RuntimeError) as e:
            continue
        rows.append((r.Decision, score_pair(model, a, b, "center"), score_pair(model, a, b, "max-window")))
    y, p_center, p_window = map(np.array, zip(*rows))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
    plots.roc(y, p_center, "MCIC: center window", axes[0]); plots.roc(y, p_window, "MCIC: best window", axes[1]); plt.show()
else:
    print("Needs the checkpoint and data/mcic (see README).")

## 6. Grad-CAM: what does the model look at?

Grad-CAM back-propagates the similarity logit to the last conv layer of each branch, giving one heat map per input.

**Held-out test pair** (two versions of one segment, scored 0.85 similar). Attention lands on different, unaligned
regions in the two branches rather than following a melodic contour:

<img src="../docs/figures/test_gradcam_similar.png" width="820">

**MCIC, false positive (court: no infringement; model: similar)**, from the course run. Highlights broad timbral overlap
from the shared soundfont rendering rather than melodic material:

<img src="../docs/figures/fig5_gradcam_mcic_fp_A.png" width="420"> <img src="../docs/figures/fig5_gradcam_mcic_fp_B.png" width="420">

In [ ]:
from musicsim.gradcam import SiameseGradCAM

if HAVE_CKPT and HAVE_DATA:
    cam = SiameseGradCAM(model)
    for label, name in [(1, "similar"), (0, "dissimilar")]:
        k = next(i for i, (_, _, yy) in enumerate(test_pairs.pairs) if yy == label)
        a, b, _ = test_pairs[k]
        ha, hb, prob = cam(a, b)
        d = test_pairs.describe(k)
        plots.gradcam_pair(a, b, ha, hb, f"{name}: {d['a']} vs {d['b']}  (p_similar={prob:.2f})"); plt.show()
else:
    print("Needs the checkpoint and preprocessed MelodySim data.")

## Conclusions

1. On unseen MelodySim tracks the model separates versions of the same segment from unrelated tracks moderately well
   (ROC-AUC 0.76).
2. On real copyright cases it is at chance (ROC-AUC 0.45) and rates every case as similar; looking for the
   best-matching windows makes it worse.
3. Grad-CAM shows why: the model relies on low-level, rendition-specific spectral texture, not melodic structure.

The training objective sits at level 2 of the similarity hierarchy (*different versions of the same song*), while legal
similarity lives at levels 3–4 (*melodically similar* / *vaguely similar* songs). High in-domain scores therefore do not
transfer.

### Background: the course poster (Fall 2025)

<img src="../docs/poster/poster.png" width="900">